In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client=MultiServerMCPClient(
    {
        "travel_server":{
             "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
        }
    }
)

tools=await client.get_tools()

In [19]:
from langchain.tools import tool
from typing import Dict,Any
from tavily import TavilyClient

tavily_client=TavilyClient()
@tool
def search_web(query:str,search_number:int ,max_search_number:int)->Dict[str,Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Query must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}


In [20]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

primary_model=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
secondary_model=ChatGoogleGenerativeAI(model="gemma-4-26b-a4b-it")

In [22]:
from langchain.agents import create_agent

travel_agent=create_agent(
    model=secondary_model,
    tools=tools,
    system_prompt="""
You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin, destination and date of wedding. It is your job to think critically about the best options and always plan the travel few days before the wedding (eg.5 days or 3 days).Always use indian currency to show ticket price.Once you have found the best options, let the user know your shortlist of options.
"""
)

In [7]:
from langchain.messages import HumanMessage
res=await travel_agent.ainvoke({'messages':[HumanMessage(content="Find flight from mangalore to bengalore on date 26 june 2026")]})

Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring


In [12]:
print(res['messages'][-1].content[-1]['text'])

Since the wedding is on June 26, 2026, I have searched for flights a few days in advance (starting from June 23) to ensure you have plenty of time to prepare and settle in.

Here is my shortlist of the best flight options from Mangalore (IXE) to Bengaluru (BLR):

### ✈️ Flight Options

| Route | Departure & Arrival | Class | Price | Booking Link |
| :--- | :--- | :--- | :--- | :--- |
| **Cheapest** | | | | |
| IXE → BLR | 23/06 09:35 → 10:40 (1h 05m) | Economy | €23 | [Book Now](https://on.kiwi.com/kNH21d) |
| IXE → BLR | 26/06 08:40 → 09:45 (1h 05m) | Economy | €23 | [Book Now](https://on.kiwi.com/elryc1) |
| IXE → BLR | 26/06 17:55 → 00:35 (1h 10m) | Economy | €23 | [Book Now](https://on.kiwi.com/mf5xxm) |
| **Shortest** | | | | |
| IXE → BLR | 23/06 08:45 → 09:50 (1h 05m) | Economy | €34 | [Book Now](https://on.kiwi.com/ZOLDOd) |
| IXE → BLR | 23/06 10:35 → 11:40 (1h 05m) | Economy | €40 | [Book Now](https://on.kiwi.com/5D2FdU) |
| **Others** | | | | |
| IXE → BOM → BLR | 23/06 08:0

In [23]:
venue_agent=create_agent(
    model=secondary_model,
    tools=[search_web],
    system_prompt="""You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
     You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have
    found so far."""
)

In [24]:
playlist_agent=create_agent(
    model=primary_model,
    tools=[query_playlist_db],
    system_prompt="""You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema."""
)

In [25]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin:str
    destination:str
    guest_count:str
    genre:str
    date:str

In [26]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage,ToolMessage
from langgraph.types import Command

@tool
async def search_flight(runtime:ToolRuntime)->str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin=runtime.state['origin']
    destination=runtime.state['destination']
    date=runtime.state['date']
    response=await travel_agent.ainvoke({"messages":[HumanMessage(content=f"Find flights from {origin} to {destination}.The date of wedding is {date}")]})
    return response['messages'][-1].content[-1]['text']


@tool
def search_venue(runtime:ToolRuntime)->str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination=runtime.state['destination']
    capacity=runtime.state['guest_count']
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response=venue_agent.invoke({"messages":[HumanMessage(content=query)]})
    return response['messages'][-1].content[-1]['text']

@tool
def suggest_playlist(runtime:ToolRuntime)->str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre=runtime.state['genre']
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content[-1]['text']

@tool
def update_state(origin:str,destination:str,guest_count:str,genre:str,date:str,runtime:ToolRuntime)->Command:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(
        update={
           "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "date":date,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
    )


In [32]:
from langgraph.checkpoint.memory import InMemorySaver

coordindator=create_agent(
    model=primary_model,
    tools=[search_flight,search_venue,suggest_playlist,update_state],
    system_prompt="""You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.""",
    state_schema=WeddingState,
    checkpointer=InMemorySaver()
)

In [33]:
msg=HumanMessage(content="I'm from Mangalore and I'd like a wedding in Goa with vibrant beach ceremony for 100 guests on 28 june 2026, jazz-genre")
config={
    'configurable':{'thread_id':1}
    }

response=await coordindator.ainvoke(
    {"messages":[msg]},
    config
    )

response

e:\LX\wedding_planner\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(
e:\LX\wedding_planner\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is

{'messages': [HumanMessage(content="I'm from Mangalore and I'd like a wedding in Goa with vibrant beach ceremony for 100 guests on 28 june 2026, jazz-genre", additional_kwargs={}, response_metadata={}, id='9867a53e-05d8-4d58-b286-21fc208babf1'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_state', 'arguments': '{"guest_count": "100", "date": "2026-06-28", "destination": "Goa", "origin": "Mangalore", "genre": "jazz"}'}, '__gemini_function_call_thought_signatures__': {'RWLQuHtB': 'EjQKMgEMOdbHmJwTxsKVs3XYolvtyPLRwu5OKeXW89L7hSHdk9HEj61Vf3KIVr+0IlbEJl0i'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebc5c-03da-7962-bb8e-b3bb14705d9a-0', tool_calls=[{'name': 'update_state', 'args': {'guest_count': '100', 'date': '2026-06-28', 'destination': 'Goa', 'origin': 'Mangalore', 'genre': 'jazz'}, 'id': 'RWLQuHtB', 'type': 'tool_call'}], invalid_tool_calls=[

In [34]:
ans=response['messages'][-1].content[-1]['text']
print(ans)

Congratulations! I have gathered all the necessary information to help you plan your perfect wedding in Goa on June 28, 2026. Here is a coordinated summary for your special day:

### ✈️ Travel Arrangements
I recommend booking the **08:40 flight on June 25, 2026** (arriving at 14:15). It is a well-balanced option that allows you to arrive in Goa comfortably before your ceremony, leaving plenty of time for preparations. 
*   **Route:** Mangalore (IXE) → Bangalore (BLR) → Goa (GOI)
*   **Estimated Cost:** ₹8,808

### 🏖️ Wedding Venue
For 100 guests looking for a beautiful beach ceremony, I highly recommend:
*   **Kenilworth Resort & Spa (South Goa):** This is the best all-around venue. It easily accommodates your guest count, offers excellent value, and is well-regarded for its beach accessibility.
*   **Budgeting:** Depending on your preferences, expect to budget roughly **₹10 Lakhs** for a mid-range wedding or **₹20 Lakhs+** for a luxury experience (covering venue, rooms, and meals for 

In [36]:
response=await coordindator.ainvoke({"messages":[HumanMessage(content="Are there any direct flights from mangalore to goa on these days")]},config)
response

e:\LX\wedding_planner\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='thread_id', input_value=1, input_type=int])
  return self.__pydantic_serializer__.to_python(
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignori

{'messages': [HumanMessage(content="I'm from Mangalore and I'd like a wedding in Goa with vibrant beach ceremony for 100 guests on 28 june 2026, jazz-genre", additional_kwargs={}, response_metadata={}, id='9867a53e-05d8-4d58-b286-21fc208babf1'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_state', 'arguments': '{"guest_count": "100", "date": "2026-06-28", "destination": "Goa", "origin": "Mangalore", "genre": "jazz"}'}, '__gemini_function_call_thought_signatures__': {'RWLQuHtB': 'EjQKMgEMOdbHmJwTxsKVs3XYolvtyPLRwu5OKeXW89L7hSHdk9HEj61Vf3KIVr+0IlbEJl0i'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ebc5c-03da-7962-bb8e-b3bb14705d9a-0', tool_calls=[{'name': 'update_state', 'args': {'guest_count': '100', 'date': '2026-06-28', 'destination': 'Goa', 'origin': 'Mangalore', 'genre': 'jazz'}, 'id': 'RWLQuHtB', 'type': 'tool_call'}], invalid_tool_calls=[

In [37]:
ans=response['messages'][-1].content[-1]['text']
print(ans)

Unfortunately, there are currently **no direct commercial flights** between Mangalore (IXE) and Goa (GOI). All available routes require at least one connection, typically via Bengaluru (BLR) or Mumbai (BOM).

The options I provided involve the most efficient layovers possible to ensure your journey is as smooth as can be. The flight via Bengaluru is generally your best bet, as it offers the shortest travel time and is quite budget-friendly.

Would you like me to look into alternative travel methods, such as a train or a private car hire, to see if those might be more convenient for you?
